In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install "anndata==0.12.2" "scanpy==1.11.4" "squidpy==1.6.5"

## Import

In [ ]:
import torch

# Check if CUDA is available
print("CUDA Available:", torch.cuda.is_available())

CUDA Available: True


In [ ]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Master_Thesis/SteamBoat/examples")
cwd = os.getcwd()
print(cwd)

sys.path.append("../")
if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/SteamBoat/examples
GPU:  Tesla T4


In [ ]:
import scanpy as sc
import squidpy as sq
import pandas as pd
from tqdm.notebook import tqdm
import scipy as sp
import numpy as np
import multiprocessing
import pickle as pkl
import torch
import gc
import sklearn.metrics

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)
/usr/local/lib/python3.12/dist-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [ ]:
import spatialdata as sd
import steamboat as sf
import steamboat.model as sm
#importlib.reload(sm)
#import steamboat.integrated_model
# importlib.reload(spaceformer.benchmarks)

In [ ]:
import importlib
import steamboat.tools
import steamboat.model

In [ ]:
importlib.reload(sf)
importlib.reload(sm)

<module 'steamboat.model' from '/content/drive/MyDrive/Thesis/Projects/Master_Thesis/SteamBoat/examples/../steamboat/model.py'>

## Creat Anndata

In [ ]:
Xenium_path = "/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr"
sdata = sd.read_zarr(Xenium_path)
sdata

/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: Use

SpatialData object, with associated Zarr store: /content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr
├── Images
│     ├── 'he_image': DataTree[cyx] (3, 24689, 17051), (3, 12344, 8525), (3, 6172, 4262), (3, 3086, 2131), (3, 1543, 1065)
│     └── 'morphology_focus': DataTree[cyx] (4, 23912, 34154), (4, 11956, 17077), (4, 5978, 8538), (4, 2989, 4269), (4, 1494, 2134)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (23912, 34154), (11956, 17077), (5978, 8538), (2989, 4269), (1494, 2134)
│     └── 'nucleus_labels': DataTree[yx] (23912, 34154), (11956, 17077), (5978, 8538), (2989, 4269), (1494, 2134)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (63173, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (63173, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (63036, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (63173, 5006)
with coordi

In [ ]:
adata = sdata.tables["table"]
adata

AnnData object with n_obs × n_vars = 63173 × 5006
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'

In [ ]:
####SINA
# to see what's inside:

adata.obs.head()
#adata.obs.columns
#adata.var["gene_symbol"].unique().shape
#adata.var["gene_symbol"].to_list()[:20]

,cell_id,transcript_counts,control_probe_counts,genomic_control_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,nucleus_area,nucleus_count,segmentation_method,region,z_level,cell_labels
0,aaaalabd-1,2193,1,0,0,3,2,2199,49.626721,33.099532,1.0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,cell_circles,0.0,1
1,aaabcnhh-1,1873,0,0,0,0,0,1873,55.226096,38.427970,1.0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,cell_circles,0.0,2
2,aaacnmip-1,2163,0,0,1,1,0,2165,85.796878,75.456096,1.0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,cell_circles,0.0,3
3,aaadikih-1,2441,0,0,0,4,1,2446,94.060472,52.290939,1.0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,cell_circles,0.0,4
4,aaaeinek-1,1407,0,0,0,0,0,1407,112.258442,NaN,0.0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,cell_circles,0.0,5


In [ ]:
####SINA
adata.var_names.to_list()[-5:]

['Zswim9', 'Zup1', 'Zyx', 'Zzef1', 'a']

In [ ]:
####SINA

adata.X[20:30, -5:].toarray()

array([[0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [0., 1., 0., 1., 0.]], dtype=float32)

In [ ]:
####SINA

adata.shape      # (cells, genes)
#adata.n_vars

(63173, 5006)

In [ ]:
####SINA

adata.uns['spatialdata_attrs']

{'instance_key': 'cell_id', 'region': 'cell_circles', 'region_key': 'region'}

In [ ]:
####SINA
for key in adata.obs.keys():
    print(f"{key}: ", "\n", adata.obs[key], "\n", "Unique Elements: ", adata.obs[key].unique().shape, "\n")

cell_id:  
 0        aaaalabd-1
1        aaabcnhh-1
2        aaacnmip-1
3        aaadikih-1
4        aaaeinek-1
            ...    
63168    oildnokf-1
63169    oildpboo-1
63170    oilgaepb-1
63171    oilgmbbn-1
63172    oilhbhom-1
Name: cell_id, Length: 63173, dtype: object 
 Unique Elements:  (63173,) 

transcript_counts:  
 0        2193
1        1873
2        2163
3        2441
4        1407
         ... 
63168     112
63169     663
63170     320
63171     299
63172     557
Name: transcript_counts, Length: 63173, dtype: int64 
 Unique Elements:  (4815,) 

control_probe_counts:  
 0        1
1        0
2        0
3        0
4        0
        ..
63168    0
63169    0
63170    0
63171    0
63172    0
Name: control_probe_counts, Length: 63173, dtype: int64 
 Unique Elements:  (3,) 

genomic_control_counts:  
 0        0
1        0
2        0
3        0
4        0
        ..
63168    0
63169    0
63170    0
63171    0
63172    0
Name: genomic_control_counts, Length: 63173, dtype: int64

## Train

In [ ]:
adata = sc.read_h5ad("../../../Data/Breast_Cancer/ann_data.h5ad")

In [ ]:
adatas = []
for i in adata.obs['region'].unique():
    adatas.append(adata[adata.obs['region'] == i])
    adatas[-1].obs['global'] = 0  #Only support one unique value for regional observation.

/tmp/ipykernel_10996/4759537.py:4: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adatas[-1].obs['global'] = 0  #Only support one unique value for regional observation.


In [ ]:
adatas = sf.prep_adatas(adatas, norm=True, log1p=True)

  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
dataset = sf.make_dataset(adatas, sparse_graph=True, regional_obs=['global'])

Using None to mask variables. Explicitly specify `mask_var=False` to use all genes.
Using ['global'] as regional annotations.


  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import gc
torch.cuda.empty_cache()

In [ ]:
sf.set_random_seed(0)
model = sm.Steamboat(adatas[0].var_names.tolist(), n_heads=64, n_scales=3)
model = model.to(device)
# model.load_state_dict(torch.load('saved_models/mmbrain_new.pth', weights_only=True))


In [ ]:
# masking_rate=0.8
model.fit(dataset, entry_masking_rate=0.8, feature_masking_rate=0,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          sched=torch.optim.lr_scheduler.OneCycleLR,
          #sched= None,
          max_lr=0.1, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=200, stop_tol=200)

[2026-05-10 19:59:22,801::train::INFO] Epoch 1: train_loss 0.26787
INFO:train:Epoch 1: train_loss 0.26787
[2026-05-10 20:00:15,941::train::INFO] Epoch 201: train_loss 0.21945
INFO:train:Epoch 201: train_loss 0.21945
[2026-05-10 20:01:09,881::train::INFO] Epoch 401: train_loss 0.14176
INFO:train:Epoch 401: train_loss 0.14176
[2026-05-10 20:02:03,628::train::INFO] Epoch 601: train_loss 0.12700
INFO:train:Epoch 601: train_loss 0.12700
[2026-05-10 20:02:57,385::train::INFO] Epoch 801: train_loss 0.12293
INFO:train:Epoch 801: train_loss 0.12293
[2026-05-10 20:03:51,092::train::INFO] Epoch 1001: train_loss 0.12022
INFO:train:Epoch 1001: train_loss 0.12022
[2026-05-10 20:04:45,053::train::INFO] Epoch 1201: train_loss 0.11841
INFO:train:Epoch 1201: train_loss 0.11841
[2026-05-10 20:05:38,858::train::INFO] Epoch 1401: train_loss 0.11706
INFO:train:Epoch 1401: train_loss 0.11706
[2026-05-10 20:06:32,550::train::INFO] Epoch 1601: train_loss 0.11641
INFO:train:Epoch 1601: train_loss 0.11641
[2026-

Steamboat(
  (spatial_gather): BilinearAttention(
    (bias): NonNegBias(
      (elu): ELU(alpha=1.0)
    )
    (q): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_regionals): ModuleList(
      (0): NonNegLinear(
        (elu): ELU(alpha=1.0)
      )
    )
    (w_ego): NonNegScale(
      (elu): ELU(alpha=1.0)
    )
    (w_local): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (w_global): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (tanh): Tanh()
    (v): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (cosine_similarity): CosineSimilarity()
  )
)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
torch.save(model.state_dict(), 'saved_models/Breast_Cancer_64_0.8.pth')

In [ ]:
sf.set_random_seed(0)
model = sm.Steamboat(adatas[0].var_names.tolist(), n_heads=64, n_scales=3)
model = model.to(device)

In [ ]:
# masking_rate=0.2

model.fit(dataset, entry_masking_rate=0.2, feature_masking_rate=0,
          device=device,
          max_epoch=5000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          sched=torch.optim.lr_scheduler.OneCycleLR,
          #sched= None,
          max_lr=0.1, opt_args=dict(lr=0.1), stop_eps=1e-7, report_per=200, stop_tol=200)

[2026-05-10 20:18:59,665::train::INFO] Epoch 1: train_loss 0.26780
INFO:train:Epoch 1: train_loss 0.26780
[2026-05-10 20:20:02,738::train::INFO] Epoch 201: train_loss 0.21850
INFO:train:Epoch 201: train_loss 0.21850
[2026-05-10 20:20:57,163::train::INFO] Epoch 401: train_loss 0.13410
INFO:train:Epoch 401: train_loss 0.13410
[2026-05-10 20:21:53,085::train::INFO] Epoch 601: train_loss 0.11668
INFO:train:Epoch 601: train_loss 0.11668
[2026-05-10 20:22:48,125::train::INFO] Epoch 801: train_loss 0.11124
INFO:train:Epoch 801: train_loss 0.11124
[2026-05-10 20:23:42,413::train::INFO] Epoch 1001: train_loss 0.10728
INFO:train:Epoch 1001: train_loss 0.10728
[2026-05-10 20:24:36,805::train::INFO] Epoch 1201: train_loss 0.10325
INFO:train:Epoch 1201: train_loss 0.10325
[2026-05-10 20:25:31,154::train::INFO] Epoch 1401: train_loss 0.09940
INFO:train:Epoch 1401: train_loss 0.09940
[2026-05-10 20:26:25,524::train::INFO] Epoch 1601: train_loss 0.09639
INFO:train:Epoch 1601: train_loss 0.09639
[2026-

Steamboat(
  (spatial_gather): BilinearAttention(
    (bias): NonNegBias(
      (elu): ELU(alpha=1.0)
    )
    (q): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_regionals): ModuleList(
      (0): NonNegLinear(
        (elu): ELU(alpha=1.0)
      )
    )
    (w_ego): NonNegScale(
      (elu): ELU(alpha=1.0)
    )
    (w_local): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (w_global): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (tanh): Tanh()
    (v): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (cosine_similarity): CosineSimilarity()
  )
)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
torch.save(model.state_dict(), 'saved_models/Breast_Cancer_64_0.2.pth')

In [ ]:
sf.set_random_seed(0)
model = sm.Steamboat(adatas[0].var_names.tolist(), n_heads=64, n_scales=3)
model = model.to(device)

In [ ]:
# masking_rate=0.5

model.fit(dataset, entry_masking_rate=0.5, feature_masking_rate=0,
          device=device,
          max_epoch=5000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          sched=torch.optim.lr_scheduler.OneCycleLR,
          #sched= None,
          max_lr=0.1, opt_args=dict(lr=0.1), stop_eps=1e-7, report_per=200, stop_tol=200)

[2026-05-10 20:43:19,921::train::INFO] Epoch 1: train_loss 0.26781
INFO:train:Epoch 1: train_loss 0.26781
[2026-05-10 20:44:21,431::train::INFO] Epoch 201: train_loss 0.21879
INFO:train:Epoch 201: train_loss 0.21879
[2026-05-10 20:45:15,464::train::INFO] Epoch 401: train_loss 0.13622
INFO:train:Epoch 401: train_loss 0.13622
[2026-05-10 20:46:12,110::train::INFO] Epoch 601: train_loss 0.11951
INFO:train:Epoch 601: train_loss 0.11951
[2026-05-10 20:47:06,304::train::INFO] Epoch 801: train_loss 0.11475
INFO:train:Epoch 801: train_loss 0.11475
[2026-05-10 20:48:01,061::train::INFO] Epoch 1001: train_loss 0.11165
INFO:train:Epoch 1001: train_loss 0.11165
[2026-05-10 20:48:55,588::train::INFO] Epoch 1201: train_loss 0.10880
INFO:train:Epoch 1201: train_loss 0.10880
[2026-05-10 20:49:51,807::train::INFO] Epoch 1401: train_loss 0.10659
INFO:train:Epoch 1401: train_loss 0.10659
[2026-05-10 20:50:46,272::train::INFO] Epoch 1601: train_loss 0.10455
INFO:train:Epoch 1601: train_loss 0.10455
[2026-

Steamboat(
  (spatial_gather): BilinearAttention(
    (bias): NonNegBias(
      (elu): ELU(alpha=1.0)
    )
    (q): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_regionals): ModuleList(
      (0): NonNegLinear(
        (elu): ELU(alpha=1.0)
      )
    )
    (w_ego): NonNegScale(
      (elu): ELU(alpha=1.0)
    )
    (w_local): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (w_global): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (tanh): Tanh()
    (v): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (cosine_similarity): CosineSimilarity()
  )
)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
torch.save(model.state_dict(), 'saved_models/Breast_Cancer_64_0.5.pth')

In [ ]:
sf.set_random_seed(0)
model = sm.Steamboat(adatas[0].var_names.tolist(), n_heads=64, n_scales=3)
model = model.to(device)

In [ ]:
# masking_rate=1.0

model.fit(dataset, entry_masking_rate=1.0, feature_masking_rate=0,
          device=device,
          max_epoch=5000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          sched=torch.optim.lr_scheduler.OneCycleLR,
          #sched= None,
          max_lr=0.1, opt_args=dict(lr=0.1), stop_eps=1e-7, report_per=200, stop_tol=200)

[2026-05-10 21:07:31,461::train::INFO] Epoch 1: train_loss 0.29644
INFO:train:Epoch 1: train_loss 0.29644
[2026-05-10 21:08:30,219::train::INFO] Epoch 201: train_loss 0.29644
INFO:train:Epoch 201: train_loss 0.29644
[2026-05-10 21:08:30,224::train::INFO] Stopping criterion met.
INFO:train:Stopping criterion met.


Steamboat(
  (spatial_gather): BilinearAttention(
    (bias): NonNegBias(
      (elu): ELU(alpha=1.0)
    )
    (q): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_regionals): ModuleList(
      (0): NonNegLinear(
        (elu): ELU(alpha=1.0)
      )
    )
    (w_ego): NonNegScale(
      (elu): ELU(alpha=1.0)
    )
    (w_local): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (w_global): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (tanh): Tanh()
    (v): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (cosine_similarity): CosineSimilarity()
  )
)

In [ ]:
os.makedirs("saved_models", exist_ok=True)
torch.save(model.state_dict(), 'saved_models/Breast_Cancer_64_1.0.pth')